# Step 4 — Model Evaluation & Final Selection
  
**Purpose:** Compare every trained model (baseline *and* tuned) on the held-out test set, 
produce diagnostic plots, and programmatically choose the single best model to deploy.

---

## Why a Dedicated Evaluation Module?

Mixing evaluation logic into training notebooks leads to:
- **Cherry-picking** — it becomes tempting to re-run training until metrics look good.
- **Leakage risk** — evaluation code that lives next to training code is more likely to accidentally 
  touch the training set.
- **Reproducibility problems** — the exact comparison procedure is not pinned.

A self-contained module solves all three: it takes already-fitted objects plus a frozen test 
split and produces deterministic numbers every time.

---

## What This Notebook Covers

| Section | Function | Output |
|---|---|---|
| Imports | — | dependencies loaded |
| `_compute_metrics()` | helper | dict of RMSE / MAE / R² |
| `build_summary_table()` | aggregator | combined DataFrame + console table |
| `plot_predictions_vs_actual()` | visualiser | `predictions_vs_actual.png` |
| `plot_residuals()` | visualiser | `residuals.png` |
| `select_final_model()` | selector **(fixed)** | winner name + fitted estimator |
| `run_evaluation()` | orchestrator | calls all of the above |


## Imports

Standard scientific stack plus `sklearn` metrics.  
`matplotlib.use('Agg')` is set **before** importing `pyplot` so that the module works correctly 
in headless environments (CI servers, Docker containers, pipeline scripts) where no display is 
available.  Using the `Agg` backend means we only write PNG files — we never try to open a 
GUI window, which would crash a non-interactive process.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')  # headless / non-interactive backend — must come before pyplot import
import matplotlib.pyplot as plt
from pathlib import Path
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

## `_compute_metrics()` — Internal Metric Helper

### Design decisions

**Single responsibility.** Every public function that needs numbers calls this one helper rather 
than computing metrics inline.  That guarantees that RMSE, MAE, and R² are *always* calculated 
the same way throughout the module.

**RMSE vs MSE.** We report RMSE (root mean squared error) instead of raw MSE because RMSE is in 
the same currency as the target variable (Indian Rupees).  A model with RMSE ≈ 150 000 is 
off by roughly ₹1.5 lakh on average — a number a domain expert immediately understands.

**Leading underscore convention.** The function is prefixed with `_` to signal that it is 
a private implementation detail.  Callers outside the module (e.g., the orchestrating script) 
should call `run_evaluation()`, not reach inside and call `_compute_metrics()` directly.

**Return type: plain `dict`.** Returning a dict (rather than a named tuple or dataclass) keeps 
the helper trivially serialisable and easy to merge into DataFrames later.

In [ ]:
def _compute_metrics(model, X_test, y_test) -> dict:
    """Compute RMSE, MAE, and R² for a fitted model on the test split."""
    y_pred = model.predict(X_test)
    return {
        "RMSE": np.sqrt(mean_squared_error(y_test, y_pred)),
        "MAE":  mean_absolute_error(y_test, y_pred),
        "R2":   r2_score(y_test, y_pred),
    }

## `build_summary_table()` — Unified Performance Table

### Why combine Baseline **and** Tuned in one table?

Keeping both stages in a single DataFrame enables true **apples-to-apples side-by-side 
comparison** without switching between separate outputs.  Concretely:

- You can immediately see *by how much* hyperparameter tuning improved (or failed to improve) 
  each model.
- A single call to `.sort_values()` on any column instantly ranks all 2 × N model variants.
- Downstream artefacts (LaTeX tables, dashboards) only need one source of truth.

### Table schema

| Column | Type | Meaning |
|--------|------|---------|
| `Model` | str | Algorithm name (e.g. `RandomForest`) |
| `Stage` | str | `Baseline` or `Tuned` |
| `RMSE` | float | Root-mean-squared error (₹) — **lower is better** |
| `MAE` | float | Mean-absolute error (₹) — **lower is better** |
| `R2` | float | Coefficient of determination (0–1) — **higher is better** |

The columns are deliberately ordered `[Model, Stage, RMSE, MAE, R2]` so that the two 
identifiers appear first and the three metrics follow in decreasing 'penalty severity' 
(RMSE penalises outliers most, R² least).

In [ ]:
def build_summary_table(baseline_results, comparison_df):
    """
    Merge baseline and tuned results into a single ranked DataFrame.

    Parameters
    ----------
    baseline_results : pd.DataFrame
        Index = model name, columns include RMSE / MAE / R2.
    comparison_df : pd.DataFrame
        Index = model name, columns include RMSE_after / MAE_after / R2_after.

    Returns
    -------
    pd.DataFrame
        Columns: Model, Stage, RMSE, MAE, R2  (sorted by Model then Stage).
    """
    base = baseline_results[["RMSE", "MAE", "R2"]].copy()
    base.index.name = "Model"
    base["Stage"] = "Baseline"

    tuned = comparison_df[["RMSE_after", "MAE_after", "R2_after"]].copy()
    tuned.columns = ["RMSE", "MAE", "R2"]  # rename to canonical names
    tuned.index.name = "Model"
    tuned["Stage"] = "Tuned"

    summary = pd.concat([base, tuned]).reset_index()
    summary = summary[["Model", "Stage", "RMSE", "MAE", "R2"]]
    summary = summary.sort_values(["Model", "Stage"]).reset_index(drop=True)

    print("\nFull Model Performance Summary")
    print("=" * 65)
    print(summary.to_string(
        index=False,
        formatters={
            "RMSE": "{:,.0f}".format,
            "MAE":  "{:,.0f}".format,
            "R2":   "{:.4f}".format,
        },
    ))
    print("=" * 65)
    return summary

## `plot_predictions_vs_actual()` — Scatter Plot of Ŷ vs Y

### Reading the chart

**The diagonal (red dashed line) = perfect prediction.**  
A point that sits exactly on the line means the model predicted that car's price to the rupee. 
In practice, no model is perfect, so we look at the *spread* around the line.

**Points above the diagonal** → model over-predicted (predicted price > actual price).  
**Points below the diagonal** → model under-predicted (predicted price < actual price).

### Fan shape = heteroscedasticity

If the scatter cloud is narrow near the left (low-price cars) and fans out to the right 
(high-price cars), the model's errors are *not* constant — they grow with price.  
This is extremely common in car-price data because:

- Luxury / premium cars have more idiosyncratic features (rare colour, bespoke trim) that 
  a generic model cannot capture.
- The price scale itself is non-linear; a ₹10 lakh error on a ₹5 lakh car is catastrophic, 
  but the same error on a ₹1 crore car is minor.

Heteroscedasticity does not necessarily mean the model is *biased* — it just means that 
prediction uncertainty varies across the price range.  A log-transform of the target variable 
is the standard remedy.

### Why one sub-plot per model?

Placing all models on the same axes at the same scale makes visual comparison trivial: 
tighter clouds = better model.

In [ ]:
def plot_predictions_vs_actual(models, X_test, y_test, save_dir):
    """
    One scatter sub-plot per model: Predicted Price vs. Actual Price.

    The red dashed diagonal is the perfect-fit reference.  Points close to
    the line indicate accurate predictions; systematic deviations reveal bias;
    a fan shape (wider spread at higher prices) indicates heteroscedasticity.

    Parameters
    ----------
    models   : dict  {name: fitted_estimator}
    X_test   : array-like  feature matrix (test split)
    y_test   : array-like  true target values
    save_dir : Path  directory to write the PNG into
    """
    n = len(models)
    fig, axes = plt.subplots(1, n, figsize=(5 * n, 5), constrained_layout=True)
    if n == 1:
        axes = [axes]  
    fig.suptitle("Predicted vs. Actual Selling Price", fontsize=14, fontweight="bold")

    for ax, (name, model) in zip(axes, models.items()):
        y_pred = model.predict(X_test)
        r2 = r2_score(y_test, y_pred)

        ax.scatter(y_test, y_pred, alpha=0.35, s=12, color="steelblue", label="Predictions")

        lo = min(y_test.min(), y_pred.min())
        hi = max(y_test.max(), y_pred.max())
        ax.plot([lo, hi], [lo, hi], "r--", linewidth=1.5, label="Perfect fit")

        ax.set_xlabel("Actual Price", fontsize=10)
        ax.set_ylabel("Predicted Price", fontsize=10)
        ax.set_title(f"{name}\nR\u00b2 = {r2:.4f}", fontsize=10)
        ax.legend(fontsize=8)

    out = save_dir / "predictions_vs_actual.png"
    fig.savefig(out, dpi=150, bbox_inches="tight")
    plt.close(fig)
    print(f"  Saved: {out}")

## `plot_residuals()` — Two-Row Residual Diagnostic

Residuals are defined as **ŷ − y** (predicted minus actual).

### Row 1 — Residuals vs. Predicted Price

| Pattern | Interpretation |
|---------|----------------|
| Random scatter around 0 | ✅ Model has captured the signal well |
| Curve / arch | ❌ Non-linear relationship not modelled |
| Fan expanding right | ❌ Heteroscedasticity (errors grow with price) |
| Cluster of outliers | ❌ Specific price segment poorly modelled |

The **red dashed line at 0** is the ideal.  Every residual above the line is an 
over-prediction; every residual below is an under-prediction.

### Row 2 — Residual Histogram

| Shape | Interpretation |
|-------|----------------|
| Bell curve centred at 0 | ✅ Errors are unbiased and approximately normal |
| Right skew (long right tail) | ❌ Model systematically under-predicts high-price cars |
| Left skew | ❌ Model systematically over-predicts |
| Fat tails | ❌ More large errors than a normal distribution would suggest |

### The orange 'mean' vertical line

The orange line marks the **mean residual**.  
- If mean ≈ 0 → the model has no systematic bias.  
- If mean > 0 → the model consistently over-predicts (ŷ > y on average).  
- If mean < 0 → the model consistently under-predicts.

Even a tree-based model that captures non-linearities well can have a non-zero mean residual 
if the training set is imbalanced in price ranges.

In [ ]:
def plot_residuals(models, X_test, y_test, save_dir):
    """
    Two-row residual diagnostic grid.

    Row 0  —  residuals (ŷ − y) vs. predicted price.
              Random scatter around 0 = good; patterns = model misspecification.
    Row 1  —  histogram of residuals.
              Bell curve centred at 0 = unbiased; skew = systematic error.
              Orange line marks the mean residual.

    Parameters
    ----------
    models   : dict  {name: fitted_estimator}
    X_test   : array-like  feature matrix (test split)
    y_test   : array-like  true target values
    save_dir : Path  directory to write the PNG into
    """
    n = len(models)
    fig, axes = plt.subplots(2, n, figsize=(5 * n, 8), constrained_layout=True)
    if n == 1:
        axes = axes.reshape(2, 1)  # ensure 2-D indexing works for a single model
    fig.suptitle("Residual Analysis", fontsize=14, fontweight="bold")

    for col, (name, model) in enumerate(models.items()):
        y_pred   = model.predict(X_test)
        residuals = y_pred - y_test

        # ── Row 0: residuals vs. predicted ──────────────────────────────────
        ax0 = axes[0, col]
        ax0.scatter(y_pred, residuals, alpha=0.35, s=12, color="steelblue")
        ax0.axhline(0, color="red", linestyle="--", linewidth=1.5)
        ax0.set_xlabel("Predicted Price", fontsize=9)
        ax0.set_ylabel("Residual", fontsize=9)
        ax0.set_title(f"{name}\nResiduals vs. Predicted", fontsize=9)

        # ── Row 1: histogram ─────────────────────────────────────────────────
        ax1 = axes[1, col]
        ax1.hist(residuals, bins=40, color="steelblue", edgecolor="white", alpha=0.85)
        ax1.axvline(0, color="red",    linestyle="--", linewidth=1.5)
        ax1.axvline(
            residuals.mean(),
            color="orange", linestyle="-", linewidth=1.2,
            label=f"mean={residuals.mean():,.0f}",
        )
        ax1.set_xlabel("Residual", fontsize=9)
        ax1.set_ylabel("Count", fontsize=9)
        ax1.set_title("Residual Distribution", fontsize=9)
        ax1.legend(fontsize=8)

    out = save_dir / "residuals.png"
    fig.savefig(out, dpi=150, bbox_inches="tight")
    plt.close(fig)
    print(f"  Saved: {out}")

## `select_final_model()` — Champion Model Picker *(fixed)*

### Original logic

The original function:
1. Filters the summary table to tuned-stage rows (falls back to all rows if none exist).
2. Sorts candidates by R² descending then RMSE ascending.
3. Takes `iloc[0]` — the single best row.
4. Looks up the fitted estimator in the `models` dict.
5. Prints a banner and returns `(display_key, winner_estimator)`.

### The problem: R² and RMSE are averages — they hide spread

Two models can have identical RMSE yet very different **residual standard deviations**.  
The model with the lower residual std is *more consistent*: its errors are clustered tightly 
around their mean rather than scattered widely.  For a used-car buyer, a model that is 
consistently off by ₹50 000 is far more useful than one that is sometimes perfect and 
sometimes off by ₹2 lakh.

### The fix: add residual std computation

**Signature change** — `X_test` and `y_test` are added as optional keyword arguments 
(defaulting to `None`) so that:
- Existing call sites that do not pass test data **still work without modification**.
- New call sites (including `run_evaluation()`) can opt in to the richer output.

**Logic change** — after identifying the winner, the function iterates over every candidate, 
predicts on `X_test`, and records `residuals.std()`.  The results are printed in ascending 
order (lower = more consistent) with the winner flagged.

**Why not make std the primary selection criterion?**  
R² and RMSE are universally understood.  Using std as a *tiebreaker / robustness footnote* 
keeps the primary ranking stable and interpretable while surfacing useful extra information.

In [ ]:
def select_final_model(
    summary_df: pd.DataFrame,
    models: dict,
    X_test=None,
    y_test=None,
) -> tuple:
    """
    Choose the best model from the summary table and print a selection banner.

    Selection rule
    --------------
    Prefer Tuned rows; sort by R2 desc then RMSE asc; take the top row.

    Fix applied
    -----------
    When X_test / y_test are supplied, also compute residual std for every
    candidate and print a ranked robustness table.  This surfaces whether
    the winner's errors are consistently small or merely small *on average*.

    Parameters
    ----------
    summary_df : pd.DataFrame   columns: Model, Stage, RMSE, MAE, R2
    models     : dict           {"<Model> (<Stage>)": fitted_estimator}
    X_test     : array-like or None   feature matrix (test split)
    y_test     : array-like or None   true target values

    Returns
    -------
    (display_key: str, winner: fitted_estimator)
    """
    tuned_mask = summary_df["Stage"] == "Tuned"
    candidates = summary_df[tuned_mask] if tuned_mask.any() else summary_df

    best_row   = candidates.sort_values(["R2", "RMSE"], ascending=[False, True]).iloc[0]
    model_name  = best_row["Model"]
    stage       = best_row["Stage"]
    display_key = f"{model_name} ({stage})"
    matched_key = next((k for k in models if model_name in k and stage in k), display_key)
    winner      = models.get(matched_key)

    
    residual_stds = {}
    if X_test is not None and y_test is not None:
        for _, row in candidates.iterrows():
            mname  = row["Model"]
            mstage = row["Stage"]
            key = next((k for k in models if mname in k and mstage in k), None)
            if key and models[key] is not None:
                residuals = models[key].predict(X_test) - y_test
                residual_stds[f"{mname} ({mstage})"] = residuals.std()
    

    sep = "\u2500" * 52
    print(f"\n{sep}")
    print("  FINAL MODEL SELECTION")
    print(sep)
    print(f"  Winner : {display_key}")
    print(f"  R\u00b2     : {best_row['R2']:.4f}")
    print(f"  RMSE   : {best_row['RMSE']:>12,.0f}")
    print(f"  MAE    : {best_row['MAE']:>12,.0f}")

    
    if residual_stds:
        print(f"\n  Robustness \u2014 residual std (lower = more consistent errors):")
        for label, std in sorted(residual_stds.items(), key=lambda x: x[1]):
            marker = " <- winner" if label == display_key else ""
            print(f"    {label}: {std:,.0f}{marker}")


    print(sep)
    return display_key, winner

## `run_evaluation()` — Top-Level Orchestrator

### Role of this function

`run_evaluation()` is the single entry point that a calling script (e.g., `main.py`) 
invokes.  It:

1. **Normalises the model registry.**  
   Training notebooks store fitted objects in two separate dictionaries 
   (`fitted_models` for baseline Pipelines, `tuned_models` for bare estimators).  
   `run_evaluation()` merges them into one flat `all_models` dict keyed by 
   `"<ModelName> (Baseline)"` / `"<ModelName> (Tuned)"` — the same format used in 
   `summary_df` — so that downstream lookups are unambiguous.

2. **Extracts the inner estimator from Pipelines.**  
   Baseline models are wrapped in sklearn `Pipeline` objects (to chain preprocessing).  
   The function calls `pipeline.named_steps["model"]` to unwrap them.  A bare `except 
   AttributeError` fallback keeps the code resilient if a caller passes a bare estimator.

3. **Calls each sub-function in a logical order** that matches the natural analyst workflow:
   build the table → visualise predictions → visualise residuals → select winner.

4. **Passes `X_test` and `y_test` to `select_final_model()`** (the fixed version) so that 
   residual std is always computed when running through the full pipeline.

### Return value

Returns `(winner_name, final_model, summary_df)` so the caller can immediately:
- Persist `final_model` with `joblib.dump()`.
- Log `summary_df` to MLflow / W&B.
- Display `winner_name` in a report.

In [ ]:
def run_evaluation(
    baseline_results,
    comparison_df,
    fitted_models,
    tuned_models,
    X_test,
    y_test,
    save_dir,
):
    """
    End-to-end evaluation orchestrator.

    Parameters
    ----------
    baseline_results : pd.DataFrame  — RMSE/MAE/R2 for baseline models
    comparison_df    : pd.DataFrame  — RMSE_after/MAE_after/R2_after for tuned models
    fitted_models    : dict          — {name: sklearn Pipeline}  (baseline)
    tuned_models     : dict          — {name: fitted estimator}  (tuned)
    X_test           : array-like    — feature matrix (test split)
    y_test           : array-like    — true target values
    save_dir         : Path          — directory for PNG output

    Returns
    -------
    winner_name  : str
    final_model  : fitted estimator
    summary_df   : pd.DataFrame
    """
    save_dir.mkdir(parents=True, exist_ok=True)

    
    all_models = {}
    for name, pipeline in fitted_models.items():
        key = f"{name} (Baseline)"
        try:
            all_models[key] = pipeline.named_steps["model"]  
        except AttributeError:
            all_models[key] = pipeline                        
    for name, estimator in tuned_models.items():
        all_models[f"{name} (Tuned)"] = estimator

    print("\n--- Step 4: Comparison & Final Model Selection ---")

    print("\nBuilding performance summary table...")
    summary_df = build_summary_table(baseline_results, comparison_df)

    print("\nGenerating predictions vs. actual plots...")
    plot_predictions_vs_actual(all_models, X_test, y_test, save_dir)

    print("\nGenerating residual plots...")
    plot_residuals(all_models, X_test, y_test, save_dir)

   
    winner_name, final_model = select_final_model(summary_df, all_models, X_test, y_test)

    return winner_name, final_model, summary_df

## How to Read the Output

### Console output

```
Full Model Performance Summary
=================================================================
 Model          Stage     RMSE       MAE       R2
 RandomForest   Baseline  180,432    112,001   0.8621
 RandomForest   Tuned     161,874     98,340   0.8890
 XGBoost        Baseline  172,015    105,222   0.8744
 XGBoost        Tuned     155,003     94,111   0.8967
=================================================================

────────────────────────────────────────────────────
  FINAL MODEL SELECTION
────────────────────────────────────────────────────
  Winner : XGBoost (Tuned)
  R²     : 0.8967
  RMSE   :      155,003
  MAE    :       94,111

  Robustness — residual std (lower = more consistent errors):
    XGBoost (Tuned): 154,218 <- winner
    RandomForest (Tuned): 160,987
────────────────────────────────────────────────────
```

### Saved PNG files

| File | What to look for |
|------|------------------|
| `predictions_vs_actual.png` | Points tightly hugging the diagonal; no systematic bow above or below it |
| `residuals.png` | Row 1: shapeless cloud centred on the red line. Row 2: bell curve with orange mean near 0 |

### Decision checklist before promoting the winner

- [ ] R² > 0.85 (explains > 85 % of price variance)
- [ ] RMSE < ₹2 lakh (errors within a reasonable negotiation band)
- [ ] Mean residual ≈ 0 (no systematic over/under-prediction)
- [ ] Residual std is the lowest among all candidates
- [ ] No dramatic fan shape in the scatter plot (manageable heteroscedasticity)
